In [1]:
#@authors: Siyu (Emily) Lei, Kai Lee, Nathan Liu, Vincent Qiu

# import necessary packages
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
# read the files and create appropriate dataframes
df_places = pd.read_csv("places.csv")
df_userPlace = pd.read_csv("user_place.csv")
df_users = pd.read_csv("users.csv")

# merge all the dataframes into one big dataframe since they correspond to each other
df_merge = pd.merge(df_places, df_userPlace, on='place_id', how='left')
df_merge = pd.merge(df_merge, df_users, on='user_id', how='left')

# grab only the data that comes from new york to avoid the hassle of regional differences
df_ny = df_merge[df_merge['city'] == 'new_york']

In [ ]:
# convert the embedding into a proper format (from str to list containing floats)

# intialize empty array
embedding_avg = []

# begin loop
for item in df_ny['embedding']:
    # create the list by splitting by commas
    n = item.split(',')

    # take care of the first and last terms containing a string bracket
    n[0] = n[0][1:]
    n[len(n) - 1] = n[len(n) - 1][:len(n[len(n) - 1]) - 1]
    
    # turn every term inside the list, n, into a float
    # if this step fails, restart Jupyter notebook and run again
    jump = list(map(float, n))

    # add this corrected embedding into the empty array
    embedding_avg.append(jump)

# add in the new column containing the properly formatted embeddings
df_ny['embedding_avg'] = embedding_avg

In [4]:
# generate personal embedding profiles for each user

# create dictionary variable
cluster = {}

# begin loop
for user in df_ny['user_id']:
    # if the user is not in the dictionary, we must create a new personal embedding for them
    if user not in cluster:
        # initialize temporary variables
        grouped_vectors = np.array([0] * 1536)
        i = 0
        s = 0

        # initialize dataframes that only contain rows that pertain to the user
        tmp = df_ny[df_ny['user_id'] == user]['embedding_avg']
        status = df_ny[df_ny['user_id'] == user]['status']

        # iterate through each place a user has shown some sort of interest in 
        for list in tmp:
            # adjust the weights on a location's embedding based on a user's status towards said place
            val = 0
            if status.iloc[s] == 'dislike':
                val = -5
            elif status.iloc[s] == 'want to try':
                val = 0.5
            elif status.iloc[s] == 'fav':
                val = 10
            elif status.iloc[s] == 'visited':
                val = 1
            else:
                val = 0

            # create a new weighted embedding and add it to the cumulative embedding
            vector = np.array(list) * val
            grouped_vectors = grouped_vectors + vector

            # continue onto next iteration
            i += 1
            s += 1

        # take the average of all embeddings of a user and make this their new personal embedding
        grouped_vectors = grouped_vectors / i
        cluster[user] = grouped_vectors
        
    # if the user exists with a personal embedding, there's no need to do it again
    else:
        continue

In [ ]:
# create a dataframe of all weighted personal user embeddings
df_userEmbed = pd.DataFrame(data=cluster).T
df_userEmbed

In [ ]:
# Step 1: new embeddings
df = df_ny
df["embedding"] = df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=",") if isinstance(x, str) else x)

# Step 2: Average embedding per user
user_vectors = df.groupby("user_id")["embedding"].apply(lambda x: np.mean(x.tolist(), axis=0)).reset_index()

# Step 3: Numpy array
X = np.array(user_vectors["embedding"].tolist())

# Step 4: Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 5: PCA
all_embeddings = np.vstack(df['embedding'].apply(lambda x: np.array(eval(x)) if isinstance(x, str) else x).values)

# fit PCA and transform ALL embeddings upfront
n_components = min(64, all_embeddings.shape[1])  # Don't exceed original dims
pca = PCA(n_components=n_components)
reduced_embeddings = pca.fit_transform(all_embeddings)  # Transforms all embeddings at once

# add reduced embeddings back to DataFrame
df['pca_embedding'] = reduced_embeddings.tolist() # Store as list of arrays

user_vectors.set_index("user_id", inplace=True)


In [ ]:
# prepare weights based on status
status_weights = {
    "fav": 10.0,
    "visited": 1.0,
    "dislike": -5.0,
    "want to try": 0.5,
}

# merge df with user_place to get status info
df_merged = df.copy()
# filter for statuses we have weights for
df_merged = df_merged[df_merged["status"].isin(status_weights.keys())]

# compute Weighted Embedding per User with normalization
def weighted_mean_embedding(group):
    embeddings = np.vstack(group["pca_embedding"].values)  # Use PCA-reduced embeds
    weights = group["status"].map(status_weights).values.reshape(-1, 1)
    weighted_embeddings = embeddings * weights
    total_weight = np.sum(np.abs(weights)) + 1e-8
    return np.sum(weighted_embeddings, axis=0) / total_weight  #normalize by total weight

user_vectors = df_merged.groupby("user_id").apply(weighted_mean_embedding)

# create user matrix and IDs with centered cosine similarity
user_ids = user_vectors.index.tolist()
user_matrix = np.stack(user_vectors.values)

# Center the data (mean subtraction)
scaler = StandardScaler(with_mean=True, with_std=False)
user_matrix_centered = scaler.fit_transform(user_matrix)

# Modified similarity function with minimum common places threshold
def most_similar_users(target_user_id, top_n=5, min_common_places=5):
    if target_user_id not in user_ids:
        available_users = sorted(user_ids)[:5]
        raise ValueError(f"User {target_user_id} not found. Available users: {available_users}")

    target_idx = user_ids.index(target_user_id)

    # center the target vector using the same scaler
    target_vector = user_matrix[target_idx].reshape(1, -1)
    target_vector_centered = scaler.transform(target_vector)

    # compute centered cosine similarities
    similarities = cosine_similarity(target_vector_centered, user_matrix_centered).flatten()

    # filter out target user and sort
    all_indices = similarities.argsort()[::-1]
    top_indices = [i for i in all_indices if i != target_idx]

    # apply minimum common places filter
    valid_users = []
    for idx in top_indices:
        user_id = user_ids[idx]
        common_places = len(
            set(df[df['user_id'] == target_user_id]['place_id']) &
            set(df[df['user_id'] == user_id]['place_id'])
        )
        if common_places >= min_common_places:
            valid_users.append((idx, similarities[idx]))

        # early exit if we have enough users
        if len(valid_users) >= top_n:
            break

    # if not enough users meet threshold, show warning
    if len(valid_users) < top_n:
        print(f"Warning: Only found {len(valid_users)} users meeting minimum {min_common_places} common places")

    # sort valid users by similarity
    valid_users.sort(key=lambda x: x[1], reverse=True)
    top_indices = [x[0] for x in valid_users[:top_n]]
    similarities = [x[1] for x in valid_users[:top_n]]

    results = []

    # get target user's status counts once
    target_status_counts = df[df['user_id'] == target_user_id]['status'].value_counts().to_dict()

    for i, (idx, similarity_score) in enumerate(zip(top_indices, similarities), 1):
        similar_user_id = user_ids[idx]

        # get similar user's status counts
        similar_status_counts = df[df['user_id'] == similar_user_id]['status'].value_counts().to_dict()

        # get common places
        common_places = set(df[df['user_id'] == target_user_id]['place_id']) & \
                       set(df[df['user_id'] == similar_user_id]['place_id'])

        print(f"\nTop {i} Similar User: {similar_user_id}, Similarity: {similarity_score:.4f}")
        print(f"Common Places: {len(common_places)} (Minimum threshold: {min_common_places})")

        # calculate agreement counts
        agreement_counts = {status: 0 for status in status_weights.keys()}
        for place_id in common_places:
            target_status = df[(df['user_id'] == target_user_id) &
                             (df['place_id'] == place_id)]['status'].values[0]
            similar_status = df[(df['user_id'] == similar_user_id) &
                              (df['place_id'] == place_id)]['status'].values[0]
            if target_status == similar_status:
                agreement_counts[target_status] += 1

        print("Status Agreement Counts:")
        for status in status_weights.keys():
            target_count = target_status_counts.get(status, 0)
            similar_count = similar_status_counts.get(status, 0)
            agreements = agreement_counts[status]
            print(f"  {status}: {agreements} (out of {target_count} {target_user_id}, {similar_count} {similar_user_id})")

        results.append({
            'user_id': similar_user_id,
            'similarity': similarity_score,
            'common_places': len(common_places),
            'agreement_counts': agreement_counts
        })

    return results

print(most_similar_users("user_999"))

In [ ]:
# time to perform clustering analysis by first reducing dimensionality of the data using PCA
pca = PCA(n_components=10)
X_pca_check = pca.fit_transform(df_userEmbed)

# PCA Explained Variance
print("PCA Explained Variance Ratio for each component:")
print(np.round(pca.explained_variance_ratio_, 4))
print(f"Total Variance Explained by 10 components: {np.sum(pca.explained_variance_ratio_):.4f}")

# plot PCA 2D ScatterPlot
plt.figure(figsize=(8,6))
plt.scatter(X_pca_check[:, 0], X_pca_check[:, 1], s=5, alpha=0.5)
plt.title("PCA (First 2 Components)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid()
plt.show()

# t-SNE Visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(df_userEmbed)

# graph
plt.figure(figsize=(8,6))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], s=5, alpha=0.5)
plt.title("t-SNE Visualization of Embeddings")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.grid()
plt.show()

In [ ]:
# 1. run t-SNE
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(df_userEmbed)

# 2. run GMM clustering on t-SNE output
gmm = GaussianMixture(n_components=8, random_state=42)
gmm_labels = gmm.fit_predict(tsne_results)

# 3. visualize
plt.figure(figsize=(8,6))
sns.scatterplot(x=tsne_results[:,0], y=tsne_results[:,1], hue=gmm_labels, palette='Spectral', legend='full', s=20)
plt.title("GMM Clustering on t-SNE Embeddings")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.grid()
plt.legend(title="Cluster ID")
plt.show()

In [ ]:
# now, create a new DataFrame with user_id and GMM cluster labels
user_cluster_df = pd.DataFrame({
    'user_id': df_userEmbed.index,  # The user IDs
    'cluster': gmm_labels           # The cluster label predicted from GMM
})

# merge back with original df
df_with_clusters = pd.merge(df_ny, user_cluster_df, on='user_id', how='left')

# count number of users per cluster
cluster_sizes = df_with_clusters['cluster'].value_counts().sort_index()

print("Users per cluster:")
print(cluster_sizes)

# visualize cluster sizes
plt.figure(figsize=(8, 4))
sns.barplot(x=cluster_sizes.index, y=cluster_sizes.values, palette='Spectral')
plt.title("User Count per Cluster")
plt.xlabel("Cluster ID")
plt.ylabel("Number of Users")
plt.grid(axis='y')
plt.show()

In [ ]:
# make a folder to store all cluster CSVs
output_folder = "clusters_csv"
os.makedirs(output_folder, exist_ok=True)

# loop through each cluster and save
for cluster_id in sorted(df_with_clusters['cluster'].unique()):
    cluster_users = df_with_clusters[df_with_clusters['cluster'] == cluster_id]
    output_path = os.path.join(output_folder, f"cluster_{cluster_id}.csv")
    cluster_users.to_csv(output_path, index=False)
    print(f"Saved Cluster {cluster_id} to {output_path}")

In [ ]:
# load important data
clusters_folder = 'clusters_csv' 
clusters = sorted([f for f in os.listdir(clusters_folder) if f.endswith('.csv')])

all_cluster_summaries = []

# analyze each cluster
# begin loop
for file in clusters:
    # read in file and create appropriate dataframe
    cluster_path = os.path.join(clusters_folder, file)
    df = pd.read_csv(cluster_path)

    cluster_name = file.replace('.csv', '')

    # initialize temporary variables
    categories_list = []
    tags_list = []
    statuses = []
    cities = []

    # begin loop
    for _, row in df.iterrows():
        # Categories
        if pd.notnull(row.get('tags', None)):
            tags_raw = row['tags'].replace("{", "").replace("}", "").split(',')
            tags_clean = [t.strip().lower() for t in tags_raw if t.strip()]
            tags_list.extend(tags_clean)
        
        # Tags
        if pd.notnull(row.get('key_words', None)):
            kw_raw = row['key_words'].replace("{", "").replace("}", "").split(',')
            kw_clean = [k.strip().lower() for k in kw_raw if k.strip()]
            categories_list.extend(kw_clean)

        # Status
        if pd.notnull(row.get('status', None)):
            statuses.append(row['status'].strip().lower())

        # City
        if pd.notnull(row.get('city', None)):
            cities.append(row['city'].strip().lower())

    # summarize the data
    top_tags = Counter(tags_list).most_common(5)
    top_categories = Counter(categories_list).most_common(5)
    top_statuses = Counter(statuses).most_common()
    top_cities = Counter(cities).most_common()

    total_users = df['user_id'].nunique()
    total_places = len(df)

    favorites = statuses.count('favorite')
    visited = statuses.count('visited')

    if visited > 0:
        favorite_ratio = favorites / visited
    else:
        favorite_ratio = 0

    avg_places_per_user = total_places / total_users if total_users > 0 else 0

    # create a variable to store generated descriptions
    desc_parts = []

    # 1. Top tastes
    if top_categories:
        tastes = ", ".join([f"{cat[0]} ({cat[1]})" for cat in top_categories])
        desc_parts.append(f"They favor places involving {tastes}.")

    # 2. Status
    if top_statuses:
        status_main = max(top_statuses, key=lambda x: x[1])[0]
        desc_parts.append(f"Most users are '{status_main}' status.")

    # 3. City
    if top_cities:
        city_main = max(top_cities, key=lambda x: x[1])[0]
        desc_parts.append(f"Most users are based in {city_main.title()}.")

    # 4. Activity metrics
    desc_parts.append(f"Favorites/Visited ratio: {favorite_ratio:.2f}")
    desc_parts.append(f"Average places per user: {avg_places_per_user:.1f}")

    full_description = " ".join(desc_parts)

    # save the final report
    all_cluster_summaries.append({
        "Cluster ID": cluster_name,
        "Top Categories": ", ".join([f"{cat[0]} ({cat[1]})" for cat in top_categories]),
        "Top Tags": ", ".join([f"{tag[0]} ({tag[1]})" for tag in top_tags]),
        "Top Statuses": ", ".join([f"{status[0]} ({status[1]})" for status in top_statuses]),
        "Top Cities": ", ".join([f"{city[0]} ({city[1]})" for city in top_cities]),
        "Favorites/Visited Ratio": round(favorite_ratio, 2),
        "Average Places per User": round(avg_places_per_user, 1),
        "Interpretation": full_description
    })

# create a master dataframe
summary_df = pd.DataFrame(all_cluster_summaries)

# save dataframe into Excel
output_filename = 'final_cluster_deep_summary.xlsx'
summary_df.to_excel(output_filename, index=False)
print(f"All done! Saved full cluster interpretation to '{output_filename}'")

In [ ]:
# load the Excel file
df = pd.read_excel('final_cluster_deep_summary.xlsx')

# load the cluster CSVs
clusters_folder = 'clusters_csv'
clusters = sorted([f for f in os.listdir(clusters_folder) if f.endswith('.csv')])

# build the corpus (one document per cluster)
cluster_texts = []
cluster_names = []

# begin loop
for file in clusters:
    # read in data and create proper dataframe
    cluster_path = os.path.join(clusters_folder, file)
    df = pd.read_csv(cluster_path)

    all_tags = []

    for _, row in df.iterrows():
        if pd.notnull(row.get('tags', None)):
            tags = row['tags'].replace("{", "").replace("}", "").split(',')
            tags = [tag.strip().lower() for tag in tags if tag.strip()]
            all_tags.extend(tags)

    # merge all tags into one "document" per cluster
    doc = " ".join(all_tags)
    cluster_texts.append(doc)
    cluster_names.append(file.replace('.csv', ''))

# apply TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(cluster_texts)

# get feature names
feature_names = vectorizer.get_feature_names_out()

# find top TF-IDF tags per cluster
top_keywords_per_cluster = []

for idx, cluster_name in enumerate(cluster_names):
    tfidf_scores = tfidf_matrix[idx].toarray()[0]
    top_indices = tfidf_scores.argsort()[-10:][::-1]  # Top 10 highest scores
    top_keywords = [(feature_names[i], round(tfidf_scores[i], 4)) for i in top_indices]

    top_keywords_per_cluster.append({
        'Cluster ID': cluster_name,
        'Top Signature Tags': ", ".join([f"{kw[0]} ({kw[1]})" for kw in top_keywords])
    })

# Save to Excel
signature_df = pd.DataFrame(top_keywords_per_cluster)
signature_df.to_excel('tfidf_cluster_signatures.xlsx', index=False)
print("TF-IDF analysis complete. File saved successfully.")
